# 에이전트의 도구 사용(Tool Use)

이 노트북은 agent가 언제, 왜, 어떻게 도구(tool)를 호출하는지를 설명한다. 단순한 언어 모델은 문장을 생성할 수는 있지만, 정확한 계산이나 구조화된 데이터 집계, 제한된 검색 작업은 별도 도구가 훨씬 안정적이다. 여기서는 tool registry, tool selection, structured tool call 패턴을 단계적으로 실험한다.

## 학습 목표
- agent가 도구를 "선택"한다는 것이 무엇을 의미하는지 이해한다.
- tool registry 패턴이 왜 확장성과 안전성에 유리한지 설명할 수 있다.
- 구조화된 tool call이 자유형 문자열 호출보다 왜 디버깅하기 쉬운지 본다.
- calculator, data_tool, search_tool이 각각 어떤 문제에 적합한지 구분할 수 있다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


## 개념 설명

tool notebook도 먼저 현재 Python 환경을 확인한다. 도구 호출은 외부 API가 아니라 로컬 함수 실행이라서 더 단순해 보이지만, 잘못된 환경에서는 패키지 import나 경로 문제가 똑같이 발생할 수 있다.

- **목적**: tool 실험이 올바른 커널에서 실행되고 있는지 확인한다.
- **핵심 로직**: 프로젝트 루트 확인 전에 `sys.executable`을 출력해 현재 Python 실행 경로를 기록한다.
- **주요 파라미터/변수**:
  - `sys.executable`: 현재 커널의 실행 파일 경로이다.

이런 환경 확인은 특히 DGX와 로컬을 오갈 때 유용하다. 같은 노트북이라도 어느 환경에서 실행됐는지 알아야 결과를 재현할 수 있다.

### 읽을 때 체크할 점
- 이 셀의 출력은 그 자체보다도 **어느 단계의 가정이 맞았는지/틀렸는지**를 보여주는 신호로 읽는 것이 좋다.
- 스터디나 면접에서는 `무엇을 했다`보다 `왜 이런 단계를 따로 분리했는가`를 설명할 수 있어야 한다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

## 구현 준비

이 setup 셀은 교육용으로 확장된 tool 모듈을 불러오고, 기본 tool registry를 생성한다. 원래 `src/tools.py`에는 안전한 계산기와 날짜 파서가 들어 있고, 여기서는 `src/tools_extended.py`가 registry와 추가 도구를 제공한다. 핵심은 agent가 도구를 무작정 실행하는 것이 아니라, **등록된 도구 집합 안에서만 선택**하도록 제한된다는 점이다.

- **목적**: structured tool use 실험에 필요한 registry와 호출 스키마를 준비한다.
- **핵심 로직**: 프로젝트 루트를 맞춘 뒤 `ToolCall`, `build_default_registry`를 import하고, `registry = build_default_registry()`로 기본 도구 집합을 생성한다.
- **주요 파라미터/변수**:
  - `registry`: 사용 가능한 도구 목록과 실행 로직을 담은 중앙 등록소이다.
  - `ToolCall`: 도구 이름과 인자를 구조화해 담는 호출 객체이다.

왜 registry 패턴이 유용한가? 새 도구를 추가할 때 agent 전체 코드를 뒤흔들지 않고, registry에 항목만 등록하면 되기 때문이다. 동시에 허용되지 않은 도구 실행을 막는 안전 장치도 된다.

### 실제 구현 펼쳐보기: `SafeEvaluator`, `calculator()`, `date_parser()`, `keyword_extractor()`
```python
class SafeEvaluator(ast.NodeVisitor):
    def visit_Expression(self, node: ast.Expression) -> float:
        return self.visit(node.body)

    def visit_BinOp(self, node: ast.BinOp) -> float:
        operator_fn = ALLOWED_BINARY_OPERATORS.get(type(node.op))
        if operator_fn is None:
            raise ValueError("Unsupported operation in calculator expression.")
        return operator_fn(self.visit(node.left), self.visit(node.right))

    def visit_UnaryOp(self, node: ast.UnaryOp) -> float:
        operator_fn = ALLOWED_UNARY_OPERATORS.get(type(node.op))
        if operator_fn is None:
            raise ValueError("Unsupported unary operation in calculator expression.")
        return operator_fn(self.visit(node.operand))

    def visit_Constant(self, node: ast.Constant) -> float:
        if not isinstance(node.value, int | float):
            raise ValueError("Calculator accepts numeric constants only.")
        return float(node.value)

    def generic_visit(self, node: ast.AST) -> float:
        raise ValueError(f"Unsupported calculator syntax: {type(node).__name__}")
```

```python
def calculator(expression: str) -> dict[str, Any]:
    try:
        parsed = ast.parse(expression, mode="eval")
        result = SafeEvaluator().visit(parsed)
    except Exception as error:  # pragma: no cover - simple error normalization
        return {"ok": False, "error": str(error)}
    return {"ok": True, "result": int(result) if result.is_integer() else round(result, 3)}
```

```python
def date_parser(text: str, reference_date: str = DEFAULT_REFERENCE_DATE) -> dict[str, Any]:
    normalized = normalize_text(text).lower()
    anchor = datetime.fromisoformat(reference_date).date()

    if normalized == "last quarter":
        quarter = (anchor.month - 1) // 3
        year = anchor.year if quarter > 0 else anchor.year - 1
        quarter = quarter if quarter > 0 else 4
        start_month = (quarter - 1) * 3 + 1
        end_month = start_month + 2
        start = date(year, start_month, 1)
        end = date(year, end_month, calendar.monthrange(year, end_month)[1])
        return {"ok": True, "normalized": {"start": start.isoformat(), "end": end.isoformat()}}

    try:
        parsed = date_parser_lib.parse(text, fuzzy=True, default=datetime(anchor.year, 1, 1))
    except (ValueError, OverflowError):
        return {"ok": False, "error": f"Could not parse date from: {text}"}
    return {"ok": True, "normalized": parsed.date().isoformat()}
```

```python
def keyword_extractor(text: str, limit: int = 5) -> dict[str, Any]:
    keywords = Counter(content_tokens(text))
    return {"ok": True, "keywords": [token for token, _ in keywords.most_common(limit)]}
```

왜 `eval()`을 쓰지 않는가:
- `eval("__import__('os').system('rm -rf /')")` 같은 위험한 코드 실행이 가능해진다.
- AST 기반 평가기는 허용한 노드만 계산하므로 공격 표면이 훨씬 작다.

💡 면접 포인트: tool use 설계에서 정확도만큼 중요한 것이 보안이며, SafeEvaluator는 그 최소 안전장치다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.tools_extended import ToolCall, build_default_registry

pd.set_option('display.max_colwidth', 140)
registry = build_default_registry()


## 도구 목록과 선택 기준

agent가 도구를 쓴다는 말은, 모든 질문에 도구를 붙인다는 뜻이 아니다. 먼저 현재 어떤 도구가 있는지 알아야 하고, 그다음 질문에 맞는 도구만 고르는 선택(selection) 단계가 필요하다. 이 셀은 registry에 등록된 도구 목록을 먼저 확인한다.

- **목적**: agent가 사용할 수 있는 도구 공간(tool affordance)을 확인한다.
- **핵심 로직**: `registry.list_tools()`가 등록된 도구 이름과 메타데이터를 반환한다.
- **주요 파라미터/변수**:
  - `calculator`: 정밀 산술과 간단한 파생값 계산에 적합하다.
  - `data_tool`: 작은 표 형태 데이터의 집계에 적합하다.
  - `search_tool`: 제한된 텍스트 집합 안에서 관련 문장을 찾는 데 적합하다.

여기서 한 가지 더 짚고 갈 점이 있다. 기존 `src/tools.py`의 계산기는 `eval()`을 쓰지 않고 AST 기반 SafeEvaluator를 사용한다. 이유는 보안 때문이다. 자유형 Python 표현식을 그대로 실행하면 임의 코드 실행 위험이 생기므로, 허용된 산술 연산만 파싱해 계산한다.

또한 날짜 처리는 보통 `dateutil`을 쓰는데, 다양한 날짜 문자열 표현을 비교적 안정적으로 파싱하기 위해서다. 즉, tool 선택은 단순 편의 기능이 아니라 정확성과 안전성을 높이는 설계다.

### 실제 구현 펼쳐보기: `ToolRegistry`
```python
class ToolRegistry:
    def __init__(self) -> None:
        self._handlers: dict[str, ToolHandler] = {}

    def register(self, name: str, handler: ToolHandler) -> None:
        self._handlers[name] = handler

    def list_tools(self) -> list[str]:
        return sorted(self._handlers)

    def call(self, tool_call: ToolCall) -> ToolResult:
        handler = self._handlers.get(tool_call.tool_name)
        if handler is None:
            return ToolResult(tool_call.tool_name, False, {"error": "Unknown tool"})
        output = handler(**tool_call.args)
        return ToolResult(tool_call.tool_name, bool(output.get("ok", False)), output)

    def select_tools(self, query: str) -> list[str]:
        normalized = normalize_text(query).lower()
        selected: list[str] = []
        if any(marker in normalized for marker in ("calculate", "sum", "difference", "days", "total")):
            selected.append("calculator")
        if any(marker in normalized for marker in ("average", "count", "dataset", "rows", "unique")):
            selected.append("data_tool")
        if any(marker in normalized for marker in ("search", "find", "lookup", "which document")):
            selected.append("search_tool")
        return selected or ["search_tool"]
```

왜 registry 패턴이 유용한가:
- 새 도구 추가 시 `register()` 한 줄이면 된다.
- 도구 목록, 호출, 선택 로직이 한 객체 안에 모여 있어 inspectable하다.
- plugin-like 확장이 가능하다.


In [ ]:
registry.list_tools()


## 구조화된 도구 호출(Structured Tool Calls)

도구 호출을 문자열 한 줄로 흘려보내면, 나중에 어떤 인자가 전달됐는지 추적하기 어렵다. 그래서 `ToolCall(name, params)`처럼 구조화된 호출 객체를 사용한다. 이렇게 하면 intent와 execution이 분리되어, agent가 먼저 "무슨 도구를 어떤 인자로 호출할지" 명시한 뒤 실행할 수 있다.

- **목적**: 구조화된 tool call이 어떤 모양인지 확인하고, 도구별 입력이 어떻게 다른지 본다.
- **핵심 로직**: `structured_calls` 리스트에 세 종류 도구 호출을 만들고, `registry.call(tool_call)`로 순서대로 실행한 뒤 결과를 표로 정리한다.
- **주요 파라미터/변수**:
  - `ToolCall('calculator', {'expression': '12 * 3 + 4'})`: 계산 도구에 넘길 식이다.
  - `ToolCall('data_tool', {'rows': ..., 'column': 'score', 'operation': 'mean'})`: 표 데이터에서 어느 열을 어떤 집계 방식으로 볼지 정의한다.
  - `ToolCall('search_tool', {'query': 'launch date', 'documents': [...]})`: 제한된 문서 리스트 안에서 어떤 문장을 찾을지 정의한다.

이 표를 보면 같은 "tool call"이라도 입력 구조가 꽤 다르다는 점을 알 수 있다. 구조화된 호출을 쓰면 이런 차이를 코드 수준에서 명확히 표현할 수 있다.

### 실제 구현 펼쳐보기: `ToolCall`
```python
class ToolCall:
    tool_name: str
    args: dict[str, Any]

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)
```

구조화된 호출을 쓰는 이유:
- `tool_name`과 `args`가 분리되어 intent와 execution을 분리하기 쉽다.
- trace에 남길 때도 사람이 읽기 쉽다.
- function calling 스타일로 확장할 발판이 된다.


In [ ]:
structured_calls = [
    ToolCall('calculator', {'expression': '12 * 3 + 4'}),
    ToolCall('data_tool', {'rows': [{'team': 'A', 'score': 8}, {'team': 'B', 'score': 10}, {'team': 'A', 'score': 6}], 'column': 'score', 'operation': 'mean'}),
    ToolCall('search_tool', {'query': 'launch date', 'documents': ['The launch date is May 5, 2025.', 'The pilot begins in March.', 'Finance approved the budget.']}),
]
structured_results = [registry.call(tool_call).to_dict() for tool_call in structured_calls]
pd.DataFrame(structured_results)


## 도구 선택(tool selection)

도구 실행보다 앞서 중요한 것은 어떤 도구가 필요한지를 고르는 일이다. agent가 도구를 선택한다는 것은, 질문의 의도를 보고 계산이 필요한지, 작은 데이터 집계가 필요한지, 제한 검색이 필요한지를 판단한다는 뜻이다.

- **목적**: 자연어 질의에 대해 어떤 도구 조합이 적절한지 확인한다.
- **핵심 로직**: `registry.select_tools(query)`가 쿼리 문장만 보고 관련 도구 이름을 반환한다.
- **주요 파라미터/변수**:
  - `selection_examples`: 서로 다른 도구를 유도하는 예시 질문들이다.
  - `selected_tools`: registry가 추천한 도구 목록 문자열이다.

이 셀의 결과를 읽을 때는, 질문이 calculation 중심인지, text lookup 중심인지, tabular aggregation 중심인지에 따라 선택 결과가 달라지는지 보자. 좋은 도구 선택은 "가능한 모든 도구 실행"이 아니라 "필요한 도구만 최소한으로 호출"하는 것이다.

### 실제 구현 펼쳐보기: `ToolRegistry.select_tools()`와 `plan_tool_requests()`
```python
    def select_tools(self, query: str) -> list[str]:
        normalized = normalize_text(query).lower()
        selected: list[str] = []
        if any(marker in normalized for marker in ("calculate", "sum", "difference", "days", "total")):
            selected.append("calculator")
        if any(marker in normalized for marker in ("average", "count", "dataset", "rows", "unique")):
            selected.append("data_tool")
        if any(marker in normalized for marker in ("search", "find", "lookup", "which document")):
            selected.append("search_tool")
        return selected or ["search_tool"]
```

```python
def plan_tool_requests(
    query: str,
    query_type: str,
    retrieved_docs: list[dict[str, Any]],
) -> list[ToolRequest]:
    requests: list[ToolRequest] = []
    normalized = normalize_text(query).lower()
    evidence_text = " ".join(doc["text"] for doc in retrieved_docs)
    date_candidates = extract_dates(query) or extract_dates(evidence_text)

    if "how many days" in normalized or "days" in normalized or "며칠" in normalized:
        if len(date_candidates) >= 2:
            requests.append(ToolRequest("date_parser", {"text": date_candidates[0]}))
            requests.append(ToolRequest("date_parser", {"text": date_candidates[1]}))
            requests.append(ToolRequest("calculator", {"expression": "DATE_DIFF_DAYS"}))

    if query_type == "summary":
        requests.append(ToolRequest("keyword_extractor", {"text": evidence_text or query, "limit": 5}))

    return requests
```

차이점:
- `select_tools()`: 질의 텍스트만 보고 도구를 고른다.
- `plan_tool_requests()`: retrieved docs까지 참고해 더 구체적 요청을 만든다.

즉, 전자는 얕은 router이고 후자는 workflow 내부의 문맥 반영 planner에 가깝다.


In [ ]:
selection_examples = [
    'Calculate the pilot duration in days.',
    'Find which document mentions the launch date.',
    'Count how many rows are in this dataset.',
]
pd.DataFrame(
    {
        'query': selection_examples,
        'selected_tools': [', '.join(registry.select_tools(query)) for query in selection_examples],
    }
)


## 실험

이 셀은 세 도구를 각각 다시 실행해보며, 결과 형식과 용도를 비교한다. 도구 학습에서는 단순히 성공 여부보다, 각 도구가 어떤 타입의 출력을 반환하는지 이해하는 것이 중요하다. 그래야 나중에 synthesizer가 이 결과를 어떻게 소비해야 하는지도 설명할 수 있다.

- **목적**: calculator, data_tool, search_tool의 출력 차이를 나란히 비교한다.
- **핵심 로직**: `experiment_calls`를 만들고, `registry.call()` 결과를 모두 dict로 바꿔 표로 표시한다.
- **주요 파라미터/변수**:
  - `expression: '25 - 7'`: 산술 계산 예시이다.
  - `rows` / `column` / `operation`: 작은 데이터셋 평균 계산 예시이다.
  - `query='pilot window'`: 제한된 문서 리스트 안에서 관련 문장을 찾는 예시이다.

결과 표에서는 각 tool이 `output`을 어떤 형식으로 돌려주는지, 그리고 agent가 이 출력을 최종 답변에 어떻게 연결할 수 있을지를 생각해보자. 도구 사용은 모델 능력의 대체가 아니라 보완이다.

### 실제 구현 펼쳐보기: `data_tool()`과 `search_tool()`
```python
def data_tool(rows: list[dict[str, Any]], column: str, operation: str = "count") -> dict[str, Any]:
    frame = pd.DataFrame(rows)
    if column not in frame.columns:
        return {"ok": False, "error": f"Column '{column}' not found."}

    series = frame[column]
    if operation == "count":
        return {"ok": True, "operation": operation, "result": int(series.count())}
    if operation == "unique":
        return {"ok": True, "operation": operation, "result": sorted(series.dropna().astype(str).unique().tolist())}
    if operation == "mean":
        numeric = pd.to_numeric(series, errors="coerce")
        return {"ok": True, "operation": operation, "result": round(float(numeric.mean()), 3)}
    return {"ok": False, "error": f"Unsupported operation: {operation}"}
```

```python
def search_tool(query: str, documents: list[str], top_k: int = 3) -> dict[str, Any]:
    query_tokens = content_tokens(query)
    ranked: list[dict[str, Any]] = []
    for index, document in enumerate(documents, start=1):
        score = round(overlap_ratio(query_tokens, content_tokens(document)), 4)
        ranked.append(
            {
                "document_id": index,
                "text": document,
                "score": score,
            }
        )
    results = sorted(ranked, key=lambda item: item["score"], reverse=True)[:top_k]
    return {"ok": True, "results": results}
```

읽는 포인트:
- `data_tool()`은 작은 표 집계에 특화돼 있고 `count`, `unique`, `mean`만 지원한다.
- `search_tool()`은 문서 리스트 안에서 overlap 기반으로 제한 검색을 수행한다.
- 둘 다 로컬 함수라서 외부 네트워크 없이 재현 가능하다.


In [ ]:
experiment_calls = [
    ToolCall('calculator', {'expression': '25 - 7'}),
    ToolCall('data_tool', {'rows': [{'latency': 0.8}, {'latency': 1.2}, {'latency': 1.0}], 'column': 'latency', 'operation': 'mean'}),
    ToolCall('search_tool', {'query': 'pilot window', 'documents': ['Pilot window: March 10, 2025 to April 4, 2025.', 'The governance memo explains ownership.', 'The FAQ lists tool guidance.']}),
]
experiment_results = [registry.call(tool_call).to_dict() for tool_call in experiment_calls]
pd.DataFrame(experiment_results)


## 결과 해석

이 마지막 분석 셀은 각 도구가 특히 잘하는 일을 한 줄씩 정리한다. tool use 설계의 핵심은 모든 문제를 도구로 풀려는 것이 아니라, 모델이 약한 부분을 적절한 도구로 보완하는 것이다.

- **목적**: 도구별 적합한 사용처를 요약한다.
- **핵심 로직**: `analysis_frame`에 도구 이름과 best-for 설명을 저장해 표로 보여준다.
- **주요 파라미터/변수**:
  - `tool`: 비교 대상 도구 이름이다.
  - `best_for`: 해당 도구가 가장 자연스럽게 쓰이는 상황이다.

이 표를 읽을 때는 "tool이 많을수록 좋다"고 해석하지 말고, 각 tool이 충분히 분리된 책임을 가지고 있는지 보자. registry 패턴 덕분에 새 도구를 추가할 때도 기존 선택/실행 구조를 크게 흔들지 않을 수 있다.

도구를 `선택`한다는 것은 모든 질문에 도구를 붙이는 것이 아니라, 모델이 약한 지점을 정확히 골라 보완한다는 뜻이다. 계산, 날짜 파싱, 구조화된 집계처럼 **언어 생성보다 도구가 더 안정적인 영역**에만 도구를 쓰는 것이 핵심이다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'tool': 'calculator', 'best_for': 'precise arithmetic and simple derived values'},
        {'tool': 'data_tool', 'best_for': 'small structured datasets and aggregates'},
        {'tool': 'search_tool', 'best_for': 'finding relevant text before synthesis'},
    ]
)
analysis_frame


## 핵심 정리

이 노트북을 통해 tool use는 단순한 기능 추가가 아니라, agent의 약점을 안전하고 구조화된 방식으로 보완하는 설계라는 점을 확인했다. calculator는 정밀 계산, data_tool은 작은 표 집계, search_tool은 제한 검색에 각각 강하다.

또한 tool registry와 structured tool call 패턴을 쓰면, 도구 선택과 실행이 분리되어 디버깅과 확장이 쉬워진다. SafeEvaluator처럼 AST 기반 계산기를 쓰는 이유도 결국 안전성과 제어 가능성 때문이다.

💡 면접 포인트: "agent가 도구를 선택한다는 것은 모델이 모든 문제를 직접 풀지 않고, 어떤 작업을 외부 기능에 위임할지 판단하는 제어 능력을 가진다는 뜻"이라고 설명하면 좋다.

면접에서는 `보안(SafeEvaluator)`, `확장성(Registry)`, `설명 가능성(Structured ToolCall)` 세 축으로 tool use 설계를 정리하면 전달력이 좋다.
